# Titanic Modeling

This notebook fits a model pipeline, compares candidate model results, and saves the best artifact in `models/best_titanic_pipeline.joblib`.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

os.makedirs('charts', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('models', exist_ok=True)

df = pd.read_csv('titanic.csv')
target = 'survived'
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

preprocess = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])

In [ ]:
models = {
    'logistic_regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'decision_tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
}

results = []
for name, model in models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds, zero_division=0),
        'f1': f1_score(y_test, preds, zero_division=0),
        'roc_auc': roc_auc_score(y_test, probs),
    }
    results.append(metrics)

results_df = pd.DataFrame(results)
results_df.to_csv('outputs/classification_metrics.csv', index=False)
print(results_df)

In [ ]:
best_model_name = results_df.sort_values('f1', ascending=False).iloc[0]['model']
best_model = models[best_model_name]
best_pipe = Pipeline([('preprocess', preprocess), ('model', best_model)])
best_pipe.fit(X_train, y_train)
joblib.dump(best_pipe, 'models/best_titanic_pipeline.joblib')
print(f'Best model: {best_model_name}')

# Save class imbalance comparison
train_summary = pd.DataFrame({
    'actual': y_train.value_counts().sort_index(),
    'proportion': y_train.value_counts(normalize=True).sort_index()
})
train_summary.to_csv('outputs/imbalance_comparison.csv')

# Save a decision tree plot
tree_model = models['decision_tree']
tree_pipe = Pipeline([('preprocess', preprocess), ('model', tree_model)])
tree_pipe.fit(X_train, y_train)
from sklearn.tree import plot_tree
plt.figure(figsize=(18, 10))
plot_tree(tree_pipe.named_steps['model'], filled=True, feature_names=tree_pipe.named_steps['preprocess'].get_feature_names_out(), max_depth=3)
plt.title('Decision Tree')
plt.savefig('charts/decision_tree.png', dpi=150, bbox_inches='tight')
plt.close()

# ROC comparison
plt.figure(figsize=(7, 6))
for name, model in models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=name)
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.savefig('charts/roc_curves.png', dpi=150)
plt.close()

# Regression example for fare prediction
reg_df = df[['age', 'fare', 'pclass', 'sex', 'survived']].dropna().copy()
reg_df['sex'] = (reg_df['sex'] == 'female').astype(int)
reg_X = reg_df[['age', 'pclass', 'sex']]
reg_y = reg_df['fare']
reg_model = LinearRegression()
reg_model.fit(reg_X, reg_y)
preds = reg_model.predict(reg_X)
reg_mse = mean_squared_error(reg_y, preds)
reg_r2 = r2_score(reg_y, preds)
reg_metrics = pd.DataFrame([{'model': 'linear_regression', 'mse': reg_mse, 'r2': reg_r2}])
reg_metrics.to_csv('outputs/regression_metrics.csv', index=False)

plt.figure(figsize=(7, 6))
plt.scatter(preds, reg_y - preds, alpha=0.6)
plt.axhline(0, color='black', linestyle='--')
plt.xlabel('Predicted Fare')
plt.ylabel('Residuals')
plt.title('Regression Residuals')
plt.savefig('charts/regression_residuals.png', dpi=150)
plt.close()

# Final model comparison table
comparison = results_df.copy()
comparison['selected'] = comparison['model'] == best_model_name
comparison.to_csv('outputs/model_comparison.csv', index=False)
print(comparison)

## Modeling interpretation

The model comparison confirms that the selected classifier provides the strongest balance between precision and recall for the Titanic survival task. The pipeline stores the final fitted estimator so it can be reused for later inference or deployment.